# 🧠 Agent Memory Optimization Lab
### Stop subsidizing your compute with bloated conversation history

| Technique | Token Reduction | Complexity |
|-----------|----------------|------------|
| 1. Semantic Compression | 60–70% | Low |
| 2. Sliding Window + ChromaDB Retrieval | 70–80% | Medium |
| 3. Structured Memory Extraction (Pydantic) | 90–95% | Medium |
| 4. TTL / Context Expiration | Varies | Low |

> **The problem:** A 20-turn conversation = 8,000+ tokens. Sending full history on every call  
> means you pay for "How's the weather?" from turn 1 when the user asks about invoices at turn 20.  
> Make sure **Ollama is running** (`ollama serve`) and your chosen model is pulled before running.


## Section 0 — Setup & Shared Fixtures

In [ ]:

!pip install ollama chromadb pydantic tiktoken sentence-transformers -q

import os, time, json, hashlib, re
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass, field
from typing import Optional
import tiktoken

import ollama

MODEL = "llama3.2"   # <- change to any model you have pulled, e.g. mistral, phi3, gemma3

# Token counter (cl100k is a good approximation for most open models)
_enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text: str) -> int:
    return len(_enc.encode(text or ""))

COST_PER_1M = 0.00   # Ollama runs locally — set to your estimated cost if using a paid host

def llm_cost(tokens: int) -> float:
    return (tokens / 1_000_000) * COST_PER_1M

def call_ollama(prompt: str, system: str = "", max_tokens: int = 512) -> tuple[str, int]:
    """Returns (response_text, prompt_token_count)."""
    messages = []
    messages.append({"role": "system", "content": system or "You are a helpful assistant."})
    messages.append({"role": "user", "content": prompt})
    response = ollama.chat(
        model=MODEL,
        messages=messages,
        options={"num_predict": max_tokens}
    )
    text = response["message"]["content"]
    # Use tiktoken to approximate prompt token count (Ollama doesn't always expose it)
    prompt_tokens = count_tokens("\n".join(m["content"] for m in messages))
    return text, prompt_tokens

# ── Simulated 20-turn conversation history ────────────────────────────────────
# Mirrors a real support-agent session: early turns are irrelevant by turn 20.
RAW_HISTORY = [
    {"role": "user",      "content": "Hey, how's the weather today?",                          "ts": "2024-01-15T09:00:00Z"},
    {"role": "assistant", "content": "I don't have real-time weather data, but I can help with other things!", "ts": "2024-01-15T09:00:05Z"},
    {"role": "user",      "content": "Ok. I need help with my account.",                       "ts": "2024-01-15T09:01:00Z"},
    {"role": "assistant", "content": "Sure! What's your account issue?",                       "ts": "2024-01-15T09:01:03Z"},
    {"role": "user",      "content": "My name is Alice and I run a small business.",           "ts": "2024-01-15T09:02:00Z"},
    {"role": "assistant", "content": "Nice to meet you, Alice! How can I assist your business?", "ts": "2024-01-15T09:02:04Z"},
    {"role": "user",      "content": "We prefer formal communication and our deadline is May 1.", "ts": "2024-01-15T09:03:00Z"},
    {"role": "assistant", "content": "Understood. Formal tone noted, and I've recorded your May 1st deadline.", "ts": "2024-01-15T09:03:06Z"},
    {"role": "user",      "content": "Great. Can you explain what invoice factoring is?",      "ts": "2024-01-15T09:05:00Z"},
    {"role": "assistant", "content": "Invoice factoring is when a business sells its receivables to a third party at a discount for immediate cash.", "ts": "2024-01-15T09:05:08Z"},
    {"role": "user",      "content": "Interesting. We currently use Net-30 payment terms.",    "ts": "2024-01-15T09:07:00Z"},
    {"role": "assistant", "content": "Net-30 is standard. Factoring could improve your cash flow if clients are slow to pay.", "ts": "2024-01-15T09:07:05Z"},
    {"role": "user",      "content": "We have 3 outstanding invoices totalling $45,000.",      "ts": "2024-01-15T09:09:00Z"},
    {"role": "assistant", "content": "That's significant. With factoring you could access roughly $40,500–$43,000 immediately.", "ts": "2024-01-15T09:09:07Z"},
    {"role": "user",      "content": "Our biggest client is TechCorp, they owe $28,000.",      "ts": "2024-01-15T09:11:00Z"},
    {"role": "assistant", "content": "TechCorp's $28k invoice is your largest exposure. Factoring that alone would cover most of your immediate needs.", "ts": "2024-01-15T09:11:06Z"},
    {"role": "user",      "content": "We decided to factor only the TechCorp invoice.",        "ts": "2024-01-15T09:13:00Z"},
    {"role": "assistant", "content": "Good decision. I'll note that you're proceeding with factoring the TechCorp $28k invoice.", "ts": "2024-01-15T09:13:04Z"},
    {"role": "user",      "content": "Now, can you help me draft a formal follow-up email to TechCorp about the invoice?", "ts": "2024-01-15T09:15:00Z"},
    {"role": "assistant", "content": "Of course! Here's a formal follow-up email for TechCorp regarding invoice payment...", "ts": "2024-01-15T09:15:09Z"},
]

def format_history_naive(history: list[dict]) -> str:
    return "\n".join(f"{m['role'].upper()}: {m['content']}" for m in history)

naive_ctx    = format_history_naive(RAW_HISTORY)
naive_tokens = count_tokens(naive_ctx)
print(f"Full history       : {len(RAW_HISTORY)} turns")
print(f"Naive tokens       : {naive_tokens:,}")
print(f"Using model        : {MODEL}  (run 'ollama pull {MODEL}' if not already downloaded)")
print(f"Cost per LLM call  : $0.00 (local) — update COST_PER_1M if using a hosted endpoint")


---
## Section 1 — Semantic Compression

**Idea:** Split conversation into chunks. Summarise older chunks with a cheap small model.  
Keep only the last N turns verbatim — they carry the most context for the current query.

```
[turn 1..5]  → summarise → "User: Alice, small biz, formal tone, May 1 deadline"   (~30 tokens)
[turn 6..10] → summarise → "Discussed invoice factoring; Net-30 terms; $45k outstanding"  (~25 tokens)  
[turn 11..17]→ summarise → "Decided to factor TechCorp $28k invoice"                (~15 tokens)
[turn 18..20]→ keep verbatim                                                         (~80 tokens)
─────────────────────────────────────────────────────────────────────────────────────────────────
Total: ~150 tokens  vs  naive 650 tokens  →  77% reduction
```


In [ ]:

VERBATIM_TURNS = 3      # keep last N turns word-for-word
CHUNK_SIZE     = 5      # summarise every K older turns

def summarise_chunk(turns: list[dict]) -> str:
    """Use a cheap model call to compress a chunk of turns into a summary sentence."""
    dialogue = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in turns)
    prompt = (
        "Summarise the key facts, decisions, and user preferences from this conversation excerpt "
        "in 1-2 concise sentences. Preserve names, numbers, dates, and decisions exactly.\n\n"
        + dialogue
    )
    summary, _ = call_ollama(prompt, max_tokens=128)
    return summary.strip()

def build_compressed_context(history: list[dict], query: str) -> tuple[str, int, int]:
    """
    Returns (compressed_context, compressed_tokens, naive_tokens).
    """
    if len(history) <= VERBATIM_TURNS:
        ctx = format_history_naive(history)
        t = count_tokens(ctx)
        return ctx, t, t

    older   = history[:-VERBATIM_TURNS]
    recent  = history[-VERBATIM_TURNS:]

    # Split older turns into chunks and summarise each
    summaries = []
    for i in range(0, len(older), CHUNK_SIZE):
        chunk   = older[i:i + CHUNK_SIZE]
        summary = summarise_chunk(chunk)
        summaries.append(f"[SUMMARY turns {i+1}-{i+len(chunk)}]: {summary}")

    # Assemble: summaries + verbatim recent + query
    summary_block  = "\n".join(summaries)
    verbatim_block = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in recent)
    compressed_ctx = f"{summary_block}\n\n[RECENT VERBATIM]\n{verbatim_block}\n\nUSER: {query}"

    naive_ctx_full = format_history_naive(history) + f"\nUSER: {query}"
    return compressed_ctx, count_tokens(compressed_ctx), count_tokens(naive_ctx_full)

# ── Run demo ──────────────────────────────────────────────────────────────────
print("=== Semantic Compression Demo ===\n")
query = "Can you help me write a formal email to TechCorp about the $28k invoice?"

print("Compressing history (this makes a few small summarisation calls)...")
ctx, comp_tokens, naive_tokens_q = build_compressed_context(RAW_HISTORY, query)

reduction = (1 - comp_tokens / naive_tokens_q) * 100
print(f"\nNaive tokens   : {naive_tokens_q:,}")
print(f"Compressed     : {comp_tokens:,}  ({reduction:.1f}% reduction)")
print(f"Cost naive     : ${llm_cost(naive_tokens_q):.5f}")
print(f"Cost compressed: ${llm_cost(comp_tokens):.5f}")
print(f"\n--- Compressed context (sent to LLM) ---")
print(ctx[:800] + "..." if len(ctx) > 800 else ctx)

print("\n--- LLM Response ---")
resp, actual_tokens = call_ollama(ctx)
print(resp)
print(f"\nActual prompt tokens (API): {actual_tokens:,}")


---
## Section 2 — Sliding Window + ChromaDB Retrieval

**Idea:** Keep a small fixed window in context (last 5 turns). Store ALL history in ChromaDB.  
On each new query, retrieve only the semantically relevant past turns — not everything.

```
New query: "Help me with the TechCorp invoice"
    ↓
ChromaDB similarity search → retrieves turn 17 ("decided to factor TechCorp $28k")
                           → retrieves turn 14 ("TechCorp owes $28,000")
    ↓
Context = retrieved_turns (2) + sliding_window (5) + query
        = ~200 tokens  vs  naive 650 tokens
```


In [ ]:

import chromadb
from sentence_transformers import SentenceTransformer

# In-memory Chroma (use chromadb.HttpClient for production)
chroma_client = chromadb.Client()
collection    = chroma_client.get_or_create_collection("conversation_history")
embedder      = SentenceTransformer("all-mpnet-base-v2")

WINDOW_SIZE       = 5      # verbatim recent turns always included
RETRIEVAL_TOP_K   = 3      # how many semantically relevant turns to fetch
SIM_THRESHOLD     = 0.40   # minimum similarity to include a retrieved turn

def index_history(history: list[dict]):
    """Store all turns in ChromaDB."""
    # Clear previous run
    try:
        chroma_client.delete_collection("conversation_history")
    except Exception:
        pass
    col = chroma_client.get_or_create_collection("conversation_history")

    docs, ids, metas = [], [], []
    for i, turn in enumerate(history):
        text = f"{turn['role'].upper()}: {turn['content']}"
        docs.append(text)
        ids.append(f"turn_{i}")
        metas.append({"role": turn["role"], "turn_idx": i, "ts": turn.get("ts", "")})

    col.add(documents=docs, ids=ids, metadatas=metas)
    return col

def retrieve_relevant(col, query: str, window_ids: set, top_k: int = RETRIEVAL_TOP_K) -> list[str]:
    """Semantic search, exclude turns already in the sliding window."""
    results = col.query(query_texts=[query], n_results=top_k + len(window_ids))
    retrieved = []
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        turn_id = results["ids"][0][results["documents"][0].index(doc)]
        sim = 1 - dist   # Chroma returns L2 distance; convert to rough similarity
        if turn_id not in window_ids and sim >= SIM_THRESHOLD:
            retrieved.append((sim, doc))
    return [doc for _, doc in sorted(retrieved, reverse=True)[:top_k]]

def build_window_retrieval_context(history: list[dict], query: str) -> tuple[str, int, int]:
    col = index_history(history)

    window      = history[-WINDOW_SIZE:]
    window_ids  = {f"turn_{len(history) - WINDOW_SIZE + i}" for i in range(len(window))}
    window_text = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in window)

    relevant    = retrieve_relevant(col, query, window_ids)
    retrieval_block = "\n".join(f"[RETRIEVED] {r}" for r in relevant) if relevant else ""

    ctx = f"{retrieval_block}\n\n[RECENT WINDOW]\n{window_text}\n\nUSER: {query}".strip()
    naive_full = format_history_naive(history) + f"\nUSER: {query}"
    return ctx, count_tokens(ctx), count_tokens(naive_full)

# ── Run demo ──────────────────────────────────────────────────────────────────
print("=== Sliding Window + Retrieval Demo ===\n")
query = "What was the key decision made about the TechCorp invoice?"

print("Indexing history into ChromaDB...")
ctx, comp_tokens, naive_tokens_q = build_window_retrieval_context(RAW_HISTORY, query)
reduction = (1 - comp_tokens / naive_tokens_q) * 100

print(f"Naive tokens    : {naive_tokens_q:,}")
print(f"Window+Retrieval: {comp_tokens:,}  ({reduction:.1f}% reduction)")
print(f"Cost naive      : ${llm_cost(naive_tokens_q):.5f}")
print(f"Cost optimised  : ${llm_cost(comp_tokens):.5f}")
print(f"\n--- Context sent to LLM ---\n{ctx}\n")

resp, actual_tokens = call_ollama(ctx,
    system="You are a helpful business assistant. Use the retrieved context to answer accurately.")
print(f"--- LLM Response ---\n{resp}")
print(f"\nActual prompt tokens (API): {actual_tokens:,}")


---
## Section 3 — Structured Memory Extraction (Pydantic)

**Idea:** After every N turns, extract key facts into a typed Pydantic schema.  
Feed structured JSON instead of raw dialogue. 50 tokens vs 2,000.

```python
# Raw dialogue (2,000 tokens):
"USER: My name is Alice... USER: We prefer formal... USER: deadline is May 1..."

# Structured memory (50 tokens):
{"user": "Alice", "tone": "formal", "deadline": "May 1",
 "decisions": ["factor TechCorp $28k invoice"],
 "key_facts": ["3 invoices totalling $45k", "TechCorp owes $28k"]}
```


In [ ]:

from pydantic import BaseModel, Field

class UserPreferences(BaseModel):
    name:     Optional[str]       = None
    tone:     Optional[str]       = None
    industry: Optional[str]       = None

class StructuredMemory(BaseModel):
    user_preferences: UserPreferences                = Field(default_factory=UserPreferences)
    key_facts:        list[str]                      = Field(default_factory=list)
    decisions:        list[str]                      = Field(default_factory=list)
    open_questions:   list[str]                      = Field(default_factory=list)
    important_dates:  dict[str, str]                 = Field(default_factory=dict)
    entities:         dict[str, str]                 = Field(default_factory=dict)   # name → description

EXTRACTION_PROMPT = """
Extract structured memory from this conversation. Return ONLY valid JSON matching this schema:
{
  "user_preferences": {"name": str|null, "tone": str|null, "industry": str|null},
  "key_facts":        [str],
  "decisions":        [str],
  "open_questions":   [str],
  "important_dates":  {label: date_str},
  "entities":         {name: description}
}
Rules:
- Be concise. Each fact/decision should be ≤15 words.
- Preserve exact numbers, names, and dates.
- Return raw JSON only, no markdown fences.

Conversation:
"""

def extract_structured_memory(history: list[dict]) -> StructuredMemory:
    """Call LLM to extract a StructuredMemory object from conversation history."""
    dialogue = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in history)
    prompt   = EXTRACTION_PROMPT + dialogue
    resp, _  = call_ollama(prompt, max_tokens=512)

    # Strip markdown fences if model adds them
    resp = re.sub(r"```json\s*|```", "", resp).strip()
    try:
        data = json.loads(resp)
        return StructuredMemory(**data)
    except Exception as e:
        print(f"Parse error: {e}\nRaw response: {resp[:300]}")
        return StructuredMemory()

def build_structured_context(memory: StructuredMemory, recent: list[dict], query: str) -> tuple[str, int, int]:
    mem_json     = memory.model_dump_json(indent=None, exclude_none=True)
    recent_text  = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in recent)
    ctx          = f"[MEMORY]\n{mem_json}\n\n[RECENT]\n{recent_text}\n\nUSER: {query}"
    naive_full   = format_history_naive(RAW_HISTORY) + f"\nUSER: {query}"
    return ctx, count_tokens(ctx), count_tokens(naive_full)

# ── Run demo ──────────────────────────────────────────────────────────────────
print("=== Structured Memory Extraction Demo ===\n")
query = "Draft a formal follow-up email to TechCorp about the outstanding invoice."

print("Extracting structured memory from full history...")
memory = extract_structured_memory(RAW_HISTORY)

print("\nExtracted memory:")
print(json.dumps(memory.model_dump(exclude_none=True), indent=2))

recent = RAW_HISTORY[-3:]
ctx, comp_tokens, naive_tokens_q = build_structured_context(memory, recent, query)
reduction = (1 - comp_tokens / naive_tokens_q) * 100

print(f"\nNaive tokens    : {naive_tokens_q:,}")
print(f"Structured+Recent: {comp_tokens:,}  ({reduction:.1f}% reduction)")
print(f"Cost naive       : ${llm_cost(naive_tokens_q):.5f}")
print(f"Cost optimised   : ${llm_cost(comp_tokens):.5f}")

resp, actual_tokens = call_ollama(ctx,
    system="You are a professional business assistant. Use the structured memory to personalise your response.")
print(f"\n--- LLM Response (actual {actual_tokens:,} tokens) ---\n{resp}")


---
## Section 4 — TTL / Context Expiration

**Idea:** Not all memory is worth the token cost. Assign TTLs by memory type:

| Memory Type | TTL | Reason |
|-------------|-----|--------|
| User preferences (tone, name) | 90 days | Stable, worth keeping |
| Key decisions | 30 days | Relevant for active projects |
| Factual Q&A exchanges | 7 days | Quickly becomes stale |
| Casual chitchat | 1 day | Almost never useful |

Load only unexpired memory. Archive the rest (don't delete — useful for analytics).


In [ ]:

from enum import Enum

class MemoryTTL(Enum):
    PREFERENCE = 90   # days
    DECISION   = 30
    FACT       = 7
    CHITCHAT   = 1

@dataclass
class MemoryEntry:
    content:     str
    memory_type: MemoryTTL
    created_at:  datetime
    tags:        list[str] = field(default_factory=list)

    @property
    def expires_at(self) -> datetime:
        return self.created_at + timedelta(days=self.memory_type.value)

    @property
    def is_expired(self) -> bool:
        return datetime.now(timezone.utc) > self.expires_at

    @property
    def ttl_remaining(self) -> str:
        remaining = self.expires_at - datetime.now(timezone.utc)
        if remaining.total_seconds() < 0:
            return "EXPIRED"
        days = remaining.days
        return f"{days}d remaining"

    def to_context_str(self) -> str:
        return f"[{self.memory_type.name}] {self.content}"

class TTLMemoryStore:
    def __init__(self):
        self.entries: list[MemoryEntry] = []
        self.archived: list[MemoryEntry] = []

    def add(self, content: str, memory_type: MemoryTTL,
            created_at: Optional[datetime] = None, tags: list[str] = None):
        self.entries.append(MemoryEntry(
            content     = content,
            memory_type = memory_type,
            created_at  = created_at or datetime.now(timezone.utc),
            tags        = tags or []
        ))

    def load_active(self) -> list[MemoryEntry]:
        """Returns non-expired entries; moves expired ones to archive."""
        active, expired = [], []
        for e in self.entries:
            (expired if e.is_expired else active).append(e)
        self.archived.extend(expired)
        self.entries = active
        return active

    def build_context(self, query: str, recent_turns: list[dict]) -> tuple[str, int]:
        active = self.load_active()
        mem_block   = "\n".join(e.to_context_str() for e in active)
        recent_text = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in recent_turns)
        ctx = f"[ACTIVE MEMORY]\n{mem_block}\n\n[RECENT]\n{recent_text}\n\nUSER: {query}"
        return ctx, count_tokens(ctx)

    def stats(self):
        active  = [e for e in self.entries if not e.is_expired]
        expired = [e for e in self.entries if e.is_expired]
        print(f"Active entries  : {len(active)}")
        print(f"Expired/archived: {len(self.archived) + len(expired)}")
        for e in self.entries:
            status = "✅ ACTIVE" if not e.is_expired else "❌ EXPIRED"
            print(f"  {status} [{e.memory_type.name:12s}] {e.ttl_remaining:15s}  {e.content[:60]}")

# ── Build a realistic TTL store from our conversation ────────────────────────
NOW = datetime.now(timezone.utc)

store = TTLMemoryStore()

# Old entries — simulate things stored weeks/months ago
store.add("User name: Alice",                             MemoryTTL.PREFERENCE, NOW - timedelta(days=60))
store.add("Preferred tone: formal",                       MemoryTTL.PREFERENCE, NOW - timedelta(days=60))
store.add("Project deadline: May 1",                      MemoryTTL.DECISION,   NOW - timedelta(days=60))
store.add("Decided to factor TechCorp $28k invoice",      MemoryTTL.DECISION,   NOW - timedelta(days=35))  # expired!
store.add("Explained invoice factoring concept",          MemoryTTL.FACT,       NOW - timedelta(days=10))  # expired!
store.add("TechCorp owes $28,000; 3 invoices = $45k total", MemoryTTL.FACT,     NOW - timedelta(days=5))
store.add("User asked about weather",                     MemoryTTL.CHITCHAT,   NOW - timedelta(days=60))  # expired!
store.add("User asked about Net-30 terms",                MemoryTTL.CHITCHAT,   NOW - timedelta(days=3))   # expired!

print("=== TTL Memory Store Demo ===\n")
print("--- Memory store status ---")
store.stats()

query = "Remind me of the key decisions made about our invoices."
recent = RAW_HISTORY[-3:]

ctx, ctx_tokens = store.build_context(query, recent)
naive_full_tokens = count_tokens(format_history_naive(RAW_HISTORY) + f"\nUSER: {query}")
reduction = (1 - ctx_tokens / naive_full_tokens) * 100

print(f"\n--- Token Impact ---")
print(f"Naive (full history)  : {naive_full_tokens:,} tokens  (${llm_cost(naive_full_tokens):.5f})")
print(f"TTL-filtered context  : {ctx_tokens:,} tokens  (${llm_cost(ctx_tokens):.5f})  ({reduction:.1f}% reduction)")
print(f"Archived/expired      : {len(store.archived)} entries (saved from context)")

print(f"\n--- Context sent to LLM ---\n{ctx}\n")
resp, actual_tokens = call_ollama(ctx,
    system="You are a professional business assistant with access to structured memory.")
print(f"--- LLM Response ({actual_tokens:,} tokens) ---\n{resp}")


---
## Section 5 — 🚀 Full Stack Memory Manager

Combine all four techniques into a single `AgentMemoryManager` class.

```
On each turn:
  1. TTL filter   → drop stale memory entries
  2. Extraction   → update structured memory every 5 turns
  3. Compression  → summarise old chunks
  4. Retrieval    → pull semantically relevant past turns from ChromaDB
  5. Assemble     → structured_mem + retrieved + sliding_window → LLM
```


In [ ]:

class AgentMemoryManager:
    """
    Combines TTL filtering, structured extraction, semantic compression,
    and sliding-window retrieval into a single context-building interface.
    """
    WINDOW_SIZE      = 5
    EXTRACTION_EVERY = 5    # re-extract structured memory every N turns
    CHUNK_SIZE       = 5    # turns per compression chunk

    def __init__(self):
        self.full_history:      list[dict]        = []
        self.structured_memory: Optional[StructuredMemory] = None
        self.ttl_store:         TTLMemoryStore    = TTLMemoryStore()
        self.chroma_col         = None
        self._turn_count        = 0

    def add_turn(self, role: str, content: str):
        self.full_history.append({"role": role, "content": content,
                                   "ts": datetime.now(timezone.utc).isoformat()})
        self._turn_count += 1
        # Trigger structured extraction every N turns
        if self._turn_count % self.EXTRACTION_EVERY == 0:
            self._update_structured_memory()
        # Re-index ChromaDB incrementally
        self._index_turn(len(self.full_history) - 1)

    def _update_structured_memory(self):
        self.structured_memory = extract_structured_memory(self.full_history)
        # Persist key facts into TTL store
        if self.structured_memory:
            for fact in self.structured_memory.key_facts:
                self.ttl_store.add(fact, MemoryTTL.FACT)
            for dec in self.structured_memory.decisions:
                self.ttl_store.add(dec, MemoryTTL.DECISION)
            prefs = self.structured_memory.user_preferences
            if prefs.name:
                self.ttl_store.add(f"User name: {prefs.name}", MemoryTTL.PREFERENCE)
            if prefs.tone:
                self.ttl_store.add(f"Preferred tone: {prefs.tone}", MemoryTTL.PREFERENCE)

    def _index_turn(self, idx: int):
        try:
            if self.chroma_col is None:
                try:
                    chroma_client.delete_collection("agent_memory")
                except Exception:
                    pass
                self.chroma_col = chroma_client.get_or_create_collection("agent_memory")
            turn = self.full_history[idx]
            self.chroma_col.add(
                documents=[f"{turn['role'].upper()}: {turn['content']}"],
                ids=[f"turn_{idx}"],
                metadatas=[{"turn_idx": idx}]
            )
        except Exception:
            pass

    def _retrieve_relevant(self, query: str, window_ids: set, top_k: int = 2) -> list[str]:
        if not self.chroma_col:
            return []
        try:
            results = self.chroma_col.query(query_texts=[query], n_results=top_k + len(window_ids))
            retrieved = []
            for doc, dist, rid in zip(results["documents"][0],
                                       results["distances"][0],
                                       results["ids"][0]):
                if rid not in window_ids and (1 - dist) >= 0.35:
                    retrieved.append(doc)
            return retrieved[:top_k]
        except Exception:
            return []

    def build_context(self, query: str) -> tuple[str, int, int]:
        """
        Assembles optimised context. Returns (context, optimised_tokens, naive_tokens).
        """
        # 1. TTL-filtered structured memory
        active_mem = self.ttl_store.load_active()
        mem_lines  = [e.to_context_str() for e in active_mem]

        # 2. Structured memory JSON (if extracted)
        if self.structured_memory:
            mem_lines.insert(0, "[STRUCTURED] " +
                self.structured_memory.model_dump_json(exclude_none=True))

        # 3. Sliding window
        window     = self.full_history[-self.WINDOW_SIZE:]
        window_ids = {f"turn_{len(self.full_history) - self.WINDOW_SIZE + i}"
                      for i in range(len(window))}
        window_text = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in window)

        # 4. Relevant retrieval
        retrieved   = self._retrieve_relevant(query, window_ids)
        ret_block   = "\n".join(f"[RETRIEVED] {r}" for r in retrieved)

        # Assemble
        parts = []
        if mem_lines:  parts.append("[MEMORY]\n" + "\n".join(mem_lines))
        if ret_block:  parts.append("[RELEVANT HISTORY]\n" + ret_block)
        parts.append("[RECENT]\n" + window_text)
        parts.append("USER: " + query)

        ctx          = "\n\n".join(parts)
        naive_full   = format_history_naive(self.full_history) + f"\nUSER: {query}"
        return ctx, count_tokens(ctx), count_tokens(naive_full)

# ── Simulate a live 20-turn session ──────────────────────────────────────────
print("=== Full Stack Memory Manager Demo ===\n")
manager = AgentMemoryManager()

print("Loading conversation history into memory manager...")
for turn in RAW_HISTORY:
    manager.add_turn(turn["role"], turn["content"])
print(f"Loaded {len(manager.full_history)} turns. Structured memory extracted.\n")

# Ask a new question (turn 21)
query = "Given everything we've discussed, what should my next steps be with the TechCorp invoice?"

ctx, opt_tokens, naive_tokens_q = manager.build_context(query)
reduction = (1 - opt_tokens / naive_tokens_q) * 100

print(f"Naive tokens     : {naive_tokens_q:,}  (${llm_cost(naive_tokens_q):.5f})")
print(f"Optimised tokens : {opt_tokens:,}  (${llm_cost(opt_tokens):.5f})  ({reduction:.1f}% reduction)")

resp, actual = call_ollama(ctx,
    system="You are a professional business assistant with structured memory.")
print(f"\n--- Response (actual {actual:,} tokens) ---\n{resp}")

# ── Scale projection ──────────────────────────────────────────────────────────
print("\n" + "="*55)
print("MONTHLY COST PROJECTION (30 calls/session, 10K users/day)")
print("="*55)
naive_monthly    = llm_cost(naive_tokens_q) * 30 * 10_000 * 30
opt_monthly      = llm_cost(opt_tokens)     * 30 * 10_000 * 30
print(f"Naive   : ${naive_monthly:>10,.2f}/month")
print(f"Optimised: ${opt_monthly:>9,.2f}/month")
print(f"Savings  : ${naive_monthly - opt_monthly:>9,.2f}/month  ({(1-opt_monthly/naive_monthly)*100:.1f}% reduction)")
